# 04 — Version-Compatibility-Matrix Validator

Companion notebook to `06-multi-service-version-pinning-and-environment-drift.md`. Chapter 06, Part 4
names this notebook directly: *"Notebook `04_version_compatibility_matrix_validator.ipynb` in this
course's `notebooks/` folder implements exactly this check, standalone."*

This notebook implements the **machine-readable compatibility matrix** (Part 2's wiki table, made
structured) and the **platform-release validation step** (Part 4) that reads the current release
manifest, computes a proposed new manifest, and validates it against the matrix before allowing a
commit -- exactly the check that would have caught Part 3's chatbot-rollback failure *before* it
served traffic, instead of after someone noticed the monitoring dashboard looked wrong.

Two independent, complementary checks, both from the chapter:

1. **The compatibility-matrix check** — Part 2's table (chatbot version range -> compatible
   monitoring/uploader version ranges), the machine-readable form of the wiki page.
2. **The declared-requirement check** — Part 4's manifest fields (`requires_chatbot_schema_version_gte`,
   `requires_chatbot_schema_version`), a more precise, per-service-declared alternative to a
   hand-maintained table.

Fully offline: standard library + PyYAML only. No real Azure DevOps/App Configuration calls -- the
"current manifest" store is a plain in-memory dict standing in for the Azure Storage blob / App
Configuration store the chapter describes.

In [1]:
import copy
import yaml

print("Imports OK")

Imports OK


## Step 1 — The compatibility matrix (chapter 06, Part 2, made machine-readable)

The exact three rows from the chapter's wiki table, as YAML instead of a Markdown table -- the
"advisory becomes enforced" move the chapter argues for. `chatbot_versions` is a simple `"lo-hi"` or
`"lo+"` range string; `min_uploader_version: null` means "no uploader-version dependency for this
chatbot range," matching the table's "any" cells.

In [2]:
COMPATIBILITY_MATRIX_YAML = """
- chatbot_versions: "54-55"
  min_monitoring_version: 38
  max_monitoring_version: 40
  min_uploader_version: null
  notes: "Pre-citations response envelope"
- chatbot_versions: "56-58"
  min_monitoring_version: 41
  max_monitoring_version: null
  min_uploader_version: null
  notes: "Response envelope adds citations[]; monitoring v41 requires it for groundedness scoring"
- chatbot_versions: "59+"
  min_monitoring_version: 41
  max_monitoring_version: null
  min_uploader_version: 20
  notes: "Uploader v20 adds document_group_id field monitoring's traceability view reads"
"""

compatibility_matrix = yaml.safe_load(COMPATIBILITY_MATRIX_YAML)
for row in compatibility_matrix:
    print(row)

{'chatbot_versions': '54-55', 'min_monitoring_version': 38, 'max_monitoring_version': 40, 'min_uploader_version': None, 'notes': 'Pre-citations response envelope'}
{'chatbot_versions': '56-58', 'min_monitoring_version': 41, 'max_monitoring_version': None, 'min_uploader_version': None, 'notes': 'Response envelope adds citations[]; monitoring v41 requires it for groundedness scoring'}
{'chatbot_versions': '59+', 'min_monitoring_version': 41, 'max_monitoring_version': None, 'min_uploader_version': 20, 'notes': "Uploader v20 adds document_group_id field monitoring's traceability view reads"}


In [3]:
def parse_version_range(range_str: str):
    """'54-55' -> (54, 55); '59+' -> (59, None) meaning unbounded above."""
    if range_str.endswith("+"):
        return int(range_str[:-1]), None
    lo, hi = range_str.split("-")
    return int(lo), int(hi)


def find_matrix_row(chatbot_version: int, matrix=compatibility_matrix):
    for row in matrix:
        lo, hi = parse_version_range(row["chatbot_versions"])
        if lo <= chatbot_version and (hi is None or chatbot_version <= hi):
            return row
    return None


def validate_against_matrix(manifest: dict, matrix=compatibility_matrix) -> list:
    """Returns a list of violation strings (empty list == valid)."""
    violations = []
    chatbot_version = manifest["services"]["chatbot"]["version_num"]
    monitoring_version = manifest["services"]["monitoring"]["version_num"]
    uploader_version = manifest["services"]["uploader"]["version_num"]

    row = find_matrix_row(chatbot_version)
    if row is None:
        violations.append(f"chatbot v{chatbot_version} is not covered by any known compatibility-matrix row")
        return violations

    if monitoring_version < row["min_monitoring_version"]:
        violations.append(
            f"monitoring v{monitoring_version} is below the minimum v{row['min_monitoring_version']} "
            f"required for chatbot v{chatbot_version} ({row['notes']})"
        )
    if row["max_monitoring_version"] is not None and monitoring_version > row["max_monitoring_version"]:
        violations.append(
            f"monitoring v{monitoring_version} is above the maximum v{row['max_monitoring_version']} "
            f"known-compatible with chatbot v{chatbot_version} ({row['notes']})"
        )
    if row["min_uploader_version"] is not None and uploader_version < row["min_uploader_version"]:
        violations.append(
            f"uploader v{uploader_version} is below the minimum v{row['min_uploader_version']} "
            f"required for chatbot v{chatbot_version} ({row['notes']})"
        )
    return violations


print("Matrix validator functions defined")

Matrix validator functions defined


## Step 2 — The release manifest (chapter 06, Part 4) and the declared-requirement check

The manifest shape from the chapter, as a plain Python dict (`version_num` added alongside the
chapter's string `version` field purely so this notebook can do integer range comparisons without a
separate parsing step). The declared-requirement check is the more precise sibling of the matrix
check: instead of looking up a hand-maintained table, each service **states its own** requirement on
the others directly in the manifest.

In [4]:
def make_manifest(manifest_version: int, client: str, chatbot_version: int, chatbot_schema_version: int,
                   monitoring_version: int, monitoring_requires_schema_gte: int,
                   uploader_version: int, uploader_requires_schema=None) -> dict:
    return {
        "manifest_version": manifest_version,
        "client": client,
        "services": {
            "chatbot": {
                "version": f"v{chatbot_version}", "version_num": chatbot_version,
                "image_tag": str(chatbot_version), "schema_version": chatbot_schema_version,
            },
            "monitoring": {
                "version": f"v{monitoring_version}", "version_num": monitoring_version,
                "requires_chatbot_schema_version_gte": monitoring_requires_schema_gte,
            },
            "uploader": {
                "version": f"v{uploader_version}", "version_num": uploader_version,
                "requires_chatbot_schema_version": uploader_requires_schema,
            },
        },
    }


def validate_declared_requirements(manifest: dict) -> list:
    violations = []
    chatbot = manifest["services"]["chatbot"]
    monitoring = manifest["services"]["monitoring"]
    uploader = manifest["services"]["uploader"]

    required = monitoring.get("requires_chatbot_schema_version_gte")
    if required is not None and chatbot["schema_version"] < required:
        violations.append(
            f"monitoring {monitoring['version']} requires chatbot schema_version >= {required}, "
            f"but chatbot {chatbot['version']} is schema_version {chatbot['schema_version']}"
        )

    required = uploader.get("requires_chatbot_schema_version")
    if required is not None and chatbot["schema_version"] < required:
        violations.append(
            f"uploader {uploader['version']} requires chatbot schema_version >= {required}, "
            f"but chatbot {chatbot['version']} is schema_version {chatbot['schema_version']}"
        )
    return violations


def validate_manifest(manifest: dict) -> dict:
    """Runs BOTH checks (matrix + declared-requirement) -- chapter 06 Part 4, Step 3."""
    violations = validate_against_matrix(manifest) + validate_declared_requirements(manifest)
    return {"valid": len(violations) == 0, "violations": violations}


print("Manifest builder + declared-requirement validator defined")

Manifest builder + declared-requirement validator defined


## Step 3 — The good state: chatbot v58, monitoring v41, uploader v22

This is the healthy, currently-live combination from chapter 06, Part 3's scenario, before the
rollback: chatbot v58 produces `citations[]` (schema_version 3), monitoring v41 reads it, and both
checks pass.

In [5]:
current_manifest = make_manifest(
    manifest_version=17, client="hsbc",
    chatbot_version=58, chatbot_schema_version=3,
    monitoring_version=41, monitoring_requires_schema_gte=3,
    uploader_version=22, uploader_requires_schema=None,
)

result = validate_manifest(current_manifest)
print("Current manifest:", current_manifest)
print()
print("Validation result:", result)
assert result["valid"], "the currently-live v58/v41/v22 combination must validate cleanly"
print("\nConfirmed: the current, live combination is known-good under both checks.")

Current manifest: {'manifest_version': 17, 'client': 'hsbc', 'services': {'chatbot': {'version': 'v58', 'version_num': 58, 'image_tag': '58', 'schema_version': 3}, 'monitoring': {'version': 'v41', 'version_num': 41, 'requires_chatbot_schema_version_gte': 3}, 'uploader': {'version': 'v22', 'version_num': 22, 'requires_chatbot_schema_version': None}}}

Validation result: {'valid': True, 'violations': []}

Confirmed: the current, live combination is known-good under both checks.


## Step 4 — Reproducing chapter 06 Part 3's exact failure: the chatbot v55 rollback

The chatbot ships v58, which has a formatting regression, so it's rolled back to v55 -- a fast,
routine slot-swap-back that reports success on its own terms. `propose_manifest()` implements chapter
06 Part 4, Step 2: compute the proposed new manifest as "whichever service just changed, plus the
other two **unchanged**" -- monitoring stays v41, uploader stays v22, because neither of them
actually redeployed.

In [6]:
def propose_manifest(current: dict, service: str, new_version_num: int, new_schema_version=None) -> dict:
    """Chapter 06, Part 4, Step 2: the proposed manifest = the changed service's new version,
    plus the other services' versions UNCHANGED from the current manifest."""
    proposed = copy.deepcopy(current)
    proposed["manifest_version"] = current["manifest_version"] + 1
    svc = proposed["services"][service]
    svc["version"] = f"v{new_version_num}"
    svc["version_num"] = new_version_num
    if service == "chatbot" and new_schema_version is not None:
        svc["schema_version"] = new_schema_version
    return proposed


# The chatbot's own pipeline reports this rollback a clean success -- v55's OWN contract is intact.
# v55 predates the citations[] response-envelope change (introduced in v56), so its schema_version is 2.
proposed_rollback_manifest = propose_manifest(current_manifest, "chatbot", new_version_num=55, new_schema_version=2)
print("Proposed manifest after the chatbot v55 rollback:")
print(proposed_rollback_manifest)

result = validate_manifest(proposed_rollback_manifest)
print()
print("Validation result:", result)

assert result["valid"] is False, "the v55/v41/v22 combination must fail validation"
assert len(result["violations"]) == 2, "expected both the matrix check AND the declared-requirement check to independently catch this"
for v in result["violations"]:
    print(" -", v)

Proposed manifest after the chatbot v55 rollback:
{'manifest_version': 18, 'client': 'hsbc', 'services': {'chatbot': {'version': 'v55', 'version_num': 55, 'image_tag': '58', 'schema_version': 2}, 'monitoring': {'version': 'v41', 'version_num': 41, 'requires_chatbot_schema_version_gte': 3}, 'uploader': {'version': 'v22', 'version_num': 22, 'requires_chatbot_schema_version': None}}}

Validation result: {'valid': False, 'violations': ['monitoring v41 is above the maximum v40 known-compatible with chatbot v55 (Pre-citations response envelope)', 'monitoring v41 requires chatbot schema_version >= 3, but chatbot v55 is schema_version 2']}
 - monitoring v41 is above the maximum v40 known-compatible with chatbot v55 (Pre-citations response envelope)
 - monitoring v41 requires chatbot schema_version >= 3, but chatbot v55 is schema_version 2


## Step 5 — The platform-release gate: commit only if valid, or require an explicit override

Chapter 06, Part 4, Step 3: if the proposed manifest is a known-good combination, commit it; if not,
**fail the platform release**, even though the chatbot's own deploy and smoke test both went green --
and require an explicit, logged, second-approver override to proceed anyway, for the rare legitimate
case where the combination is actually fine and the matrix just hasn't been updated yet.

In [7]:
manifest_store = {"hsbc": current_manifest}


def commit_platform_release(client: str, proposed: dict, override: bool = False, override_reason: str = None) -> dict:
    result = validate_manifest(proposed)
    if result["valid"]:
        manifest_store[client] = proposed
        return {"committed": True, "overridden": False, "violations": []}

    if override:
        if not override_reason:
            raise ValueError("an override MUST be accompanied by a logged reason -- no silent overrides")
        print(f"WARNING: committing a manifest with known violations under explicit override: {override_reason}")
        manifest_store[client] = proposed
        return {"committed": True, "overridden": True, "violations": result["violations"]}

    return {"committed": False, "overridden": False, "violations": result["violations"]}


# The v55 rollback, attempted as a normal platform release -- BLOCKED before it takes over traffic.
outcome = commit_platform_release("hsbc", proposed_rollback_manifest)
print("Attempted commit (no override):", outcome)
assert outcome["committed"] is False
assert manifest_store["hsbc"]["services"]["chatbot"]["version"] == "v58", (
    "the manifest store must still reflect the last KNOWN-GOOD state -- the bad rollback never committed"
)
print("\nConfirmed: the v55 rollback is blocked at the platform-release gate BEFORE it takes over "
      "production traffic -- this is exactly what chapter 06 says the naive per-service rollback, "
      "on its own, cannot catch.")

print()
# The rare legitimate case: an authorized engineer confirms v55/v41/v22 is actually fine (say,
# monitoring v41's citations[] read is behind a feature flag that's off for this client) and
# overrides, with a logged reason.
outcome2 = commit_platform_release(
    "hsbc", proposed_rollback_manifest, override=True,
    override_reason="citations[] groundedness scoring is feature-flagged off for hsbc; confirmed with monitoring on-call",
)
print("Attempted commit (explicit override):", outcome2)
assert outcome2["committed"] is True and outcome2["overridden"] is True
assert manifest_store["hsbc"]["services"]["chatbot"]["version"] == "v55"
print("\nConfirmed: an override commits the proposed manifest anyway, but only with a logged reason -- "
      "never silently.")

Attempted commit (no override): {'committed': False, 'overridden': False, 'violations': ['monitoring v41 is above the maximum v40 known-compatible with chatbot v55 (Pre-citations response envelope)', 'monitoring v41 requires chatbot schema_version >= 3, but chatbot v55 is schema_version 2']}

Confirmed: the v55 rollback is blocked at the platform-release gate BEFORE it takes over production traffic -- this is exactly what chapter 06 says the naive per-service rollback, on its own, cannot catch.

Attempted commit (explicit override): {'committed': True, 'overridden': True, 'violations': ['monitoring v41 is above the maximum v40 known-compatible with chatbot v55 (Pre-citations response envelope)', 'monitoring v41 requires chatbot schema_version >= 3, but chatbot v55 is schema_version 2']}

Confirmed: an override commits the proposed manifest anyway, but only with a logged reason -- never silently.


## Tying it back

- The compatibility matrix (Step 1) turns Part 2's advisory wiki table into something a pipeline can
  actually evaluate, not just a page a human might remember to check.
- `propose_manifest()` (Step 4) implements the specific rule chapter 06 insists on: a rollback is
  computed and validated **exactly the same way** as a forward deploy -- there's no special case that
  lets "going backward" skip the check, which is precisely how Part 3's v55 rollback slips through a
  naive, per-service-only view of "did my deploy succeed."
- `commit_platform_release()` (Step 5) is the release-manifest gate itself: block by default, commit
  only on a valid combination or an explicit, logged override -- never a silent auto-promote of a
  combination nothing has confirmed is safe.